# 用户特征表：检查与清洗

检查主键、缺失值、分类字段与可解释数值字段，并保留原始信息生成可审计的清洗字段。


In [1]:
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """从当前目录向上定位作品集根目录。"""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Python").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请从 KuaiRand_Pure 目录或其子目录运行。")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MYSQL_IMPORT_DIR = PROJECT_ROOT / "data" / "mysql_import"


In [2]:
#导入用户表 user_features
import pandas as pd
user_features = pd.read_csv(RAW_DIR / "user_features_pure.csv")

## 1. 检查表的基本结构，主键唯一性，是否有缺失值和重复值


In [3]:
#检查表格的基本结构
print("用户表形状：", user_features.shape)
print("用户表的索引",user_features.index)
print("用户表的列名",user_features.columns.tolist())
print(user_features.info())

#检查表格的主键是否唯一
print(
    "user_id重复数量：",
    user_features["user_id"].duplicated().sum()
)

user_features.head(10)

用户表形状： (27285, 31)
用户表的索引 RangeIndex(start=0, stop=27285, step=1)
用户表的列名 ['user_id', 'user_active_degree', 'is_lowactive_period', 'is_live_streamer', 'is_video_author', 'follow_user_num', 'follow_user_num_range', 'fans_user_num', 'fans_user_num_range', 'friend_user_num', 'friend_user_num_range', 'register_days', 'register_days_range', 'onehot_feat0', 'onehot_feat1', 'onehot_feat2', 'onehot_feat3', 'onehot_feat4', 'onehot_feat5', 'onehot_feat6', 'onehot_feat7', 'onehot_feat8', 'onehot_feat9', 'onehot_feat10', 'onehot_feat11', 'onehot_feat12', 'onehot_feat13', 'onehot_feat14', 'onehot_feat15', 'onehot_feat16', 'onehot_feat17']
<class 'pandas.DataFrame'>
RangeIndex: 27285 entries, 0 to 27284
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                27285 non-null  int64  
 1   user_active_degree     27285 non-null  str    
 2   is_lowactive_period    27285 non-null  int64  
 3   is_l

,user_id,user_active_degree,is_lowactive_period,is_live_streamer,is_video_author,follow_user_num,follow_user_num_range,fans_user_num,fans_user_num_range,friend_user_num,...,onehot_feat8,onehot_feat9,onehot_feat10,onehot_feat11,onehot_feat12,onehot_feat13,onehot_feat14,onehot_feat15,onehot_feat16,onehot_feat17
0,0,full_active,0,1,1,514,500+,150,"[100,1k)",34,...,135,6,3,0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,full_active,0,-124,1,457,"(250,500]",20,"[10,100)",3,...,283,6,2,2,1.0,0.0,1.0,0.0,0.0,0.0
2,2,full_active,0,-124,1,8,"(0,10]",26,"[10,100)",3,...,275,5,2,2,0.0,0.0,0.0,1.0,0.0,0.0
3,3,full_active,0,1,1,91,"(50,100]",2166,"[1k,5k)",14,...,118,6,2,2,1.0,0.0,0.0,0.0,0.0,0.0
4,4,full_active,0,-124,1,261,"(250,500]",10,"[10,100)",4,...,378,3,3,0,0.0,0.0,0.0,0.0,0.0,0.0
5,5,full_active,0,-124,1,15,"(10,50]",23,"[10,100)",2,...,101,5,3,0,1.0,0.0,1.0,0.0,0.0,0.0
6,6,full_active,0,-124,0,75,"(50,100]",26,"[10,100)",23,...,264,4,2,0,1.0,0.0,0.0,0.0,0.0,0.0
7,7,full_active,0,-124,1,2103,500+,56,"[10,100)",43,...,396,3,3,0,0.0,0.0,0.0,0.0,0.0,0.0
8,8,high_active,0,-124,1,265,"(250,500]",27,"[10,100)",12,...,188,3,3,0,0.0,0.0,0.0,0.0,0.0,0.0
9,9,high_active,0,-124,0,71,"(50,100]",0,0,0,...,400,5,0,0,1.0,0.0,0.0,0.0,0.0,0.0


In [4]:
#检查表格的缺失值
missing_count = user_features.isna().sum()

print("存在缺失值的字段：")
print(
    missing_count[missing_count > 0]
    .sort_values(ascending=False)
)


存在缺失值的字段：
onehot_feat4     874
onehot_feat12    714
onehot_feat13    714
onehot_feat14    714
onehot_feat15    714
onehot_feat16    714
onehot_feat17    714
dtype: int64


In [5]:
#检查表格是否有重复值
print("用户表的重复记录数：",user_features.duplicated().sum())

用户表的重复记录数： 0


## 2. 检查单个字段的有效性

In [6]:
category_cols = [
    "user_active_degree",
    "is_lowactive_period",
    "is_live_streamer",
    "is_video_author",
    "follow_user_num_range",
    "fans_user_num_range",
    "friend_user_num_range",
    "register_days_range",
]

for col in category_cols:
    print(f"\n===== {col} =====")
    print(
        user_features[col]
        .value_counts(dropna=False)
    )



===== user_active_degree =====
user_active_degree
full_active          17959
high_active           6158
middle_active         2133
2_14_day_new           635
low_active             300
single_low_active       68
30day_retention         15
day_new                 11
UNKNOWN                  6
Name: count, dtype: int64

===== is_lowactive_period =====
is_lowactive_period
0    27285
Name: count, dtype: int64

===== is_live_streamer =====
is_live_streamer
-124    21127
 1       6158
Name: count, dtype: int64

===== is_video_author =====
is_video_author
1    22276
0     5009
Name: count, dtype: int64

===== follow_user_num_range =====
follow_user_num_range
500+         5950
(10,50]      4904
(250,500]    4377
(50,100]     4004
(150,250]    3490
(100,150]    2633
(0,10]       1739
0             188
Name: count, dtype: int64

===== fans_user_num_range =====
fans_user_num_range
[10,100)        10467
[100,1k)         8664
[1,10)           5326
[1k,5k)          1460
0                1157
[5k,1w

In [7]:
number_cols = [
    "follow_user_num",
    "fans_user_num",
    "friend_user_num",
    "register_days"
]

print("数值字段基本统计：")
print(
    user_features[number_cols]
    .describe()
    .T
)

print("\n各字段负数数量：")
print(
    (user_features[number_cols] < 0)
    .sum()
)


数值字段基本统计：
                   count         mean           std   min    25%     50%  \
follow_user_num  27285.0   409.832729    717.404443   0.0   50.0   155.0   
fans_user_num    27285.0   602.727946  15810.833377   0.0   10.0    48.0   
friend_user_num  27285.0   119.298332    349.903198   0.0    3.0    16.0   
register_days    27285.0  1204.401283    680.470549  14.0  683.0  1188.0   

                    75%        max  
follow_user_num   430.0     5003.0  
fans_user_num     221.0  1986249.0  
friend_user_num    83.0     4995.0  
register_days    1701.0     3624.0  

各字段负数数量：
follow_user_num    0
fans_user_num      0
friend_user_num    0
register_days      0
dtype: int64


## 3. 检查"数值字段与分组字段"的跨字段逻辑一致性

In [13]:
# ==================================================
# 1. 粉丝数与粉丝数分组的一致性检查
# ==================================================
def get_fans_range(fans_num):
    if fans_num == 0:
        return "0"
    elif 1 <= fans_num < 10:
        return "[1,10)"
    elif 10 <= fans_num < 100:
        return "[10,100)"
    elif 100 <= fans_num < 1_000:
        return "[100,1k)"
    elif 1_000 <= fans_num < 5_000:
        return "[1k,5k)"
    elif 5_000 <= fans_num < 10_000:
        return "[5k,1w)"
    elif 10_000 <= fans_num < 100_000:
        return "[1w,10w)"
    elif 100_000 <= fans_num < 1_000_000:
        return "[10w,100w)"
    elif 1_000_000 <= fans_num < 10_000_000:
        return "[100w,1000w)"
    else:
        return "未定义范围"


# 根据粉丝数计算预期分组
expected_fans_range = (
    user_features["fans_user_num"]
    .apply(get_fans_range)
)

# 判断预期分组是否与原分组一致
fans_range_mismatch = (
    expected_fans_range
    != user_features["fans_user_num_range"]
)

print(
    "粉丝数与粉丝分组不一致的记录数：",
    fans_range_mismatch.sum()
)

# 查看异常记录
fans_range_issues = user_features.loc[
    fans_range_mismatch,
    [
        "user_id",
        "fans_user_num",
        "fans_user_num_range"
    ]
].copy()

fans_range_issues["预期粉丝分组"] = (
    expected_fans_range.loc[fans_range_mismatch]
)

display(fans_range_issues)

# ==================================================
# 2. 关注数与关注数分组的一致性检查
# ==================================================

def get_follow_range(follow_num):
    if follow_num == 0:
        return "0"
    elif 0 < follow_num <= 10:
        return "(0,10]"
    elif 10 < follow_num <= 50:
        return "(10,50]"
    elif 50 < follow_num <= 100:
        return "(50,100]"
    elif 100 < follow_num <= 150:
        return "(100,150]"
    elif 150 < follow_num <= 250:
        return "(150,250]"
    elif 250 < follow_num <= 500:
        return "(250,500]"
    elif follow_num > 500:
        return "500+"
    else:
        return "未定义范围"


# 根据关注数计算预期分组
expected_follow_range = (
    user_features["follow_user_num"]
    .apply(get_follow_range)
)

# 判断预期分组是否与原分组一致
follow_range_mismatch = (
    expected_follow_range
    != user_features["follow_user_num_range"]
)

print(
    "关注数与关注数分组不一致的记录数：",
    follow_range_mismatch.sum()
)

# 查看异常记录
follow_range_issues = user_features.loc[
    follow_range_mismatch,
    [
        "user_id",
        "follow_user_num",
        "follow_user_num_range"
    ]
].copy()

follow_range_issues["预期关注数分组"] = (
    expected_follow_range.loc[follow_range_mismatch]
)

display(follow_range_issues)


# ==================================================
# 2. 朋友数与朋友数分组的一致性检查
# ==================================================

def get_friend_range(friend_num):
    if friend_num == 0:
        return "0"
    elif 1 <= friend_num < 5:
        return "[1,5)"
    elif 5 <= friend_num < 30:
        return "[5,30)"
    elif 30 <= friend_num < 60:
        return "[30,60)"
    elif 60 <= friend_num < 120:
        return "[60,120)"
    elif 120 <= friend_num < 250:
        return "[120,250)"
    elif friend_num >= 250:
        return "250+"
    else:
        return "未定义范围"


# 根据朋友数计算预期分组
expected_friend_range = (
    user_features["friend_user_num"]
    .apply(get_friend_range)
)

# 判断预期分组是否与原分组一致
friend_range_mismatch = (
    expected_friend_range
    != user_features["friend_user_num_range"]
)

print(
    "朋友数与朋友数分组不一致的记录数：",
    friend_range_mismatch.sum()
)

# 查看异常记录
friend_range_issues = user_features.loc[
    friend_range_mismatch,
    [
        "user_id",
        "friend_user_num",
        "friend_user_num_range"
    ]
].copy()

friend_range_issues["预期朋友数分组"] = (
    expected_friend_range.loc[friend_range_mismatch]
)

display(friend_range_issues)


# ==================================================
# 3. 注册天数与注册天数分组的一致性检查
# ==================================================

def get_register_days_range(register_days):
    if 8 <= register_days <= 14:
        return "8-14"
    elif 15 <= register_days <= 30:
        return "15-30"
    elif 31 <= register_days <= 60:
        return "31-60"
    elif 61 <= register_days <= 90:
        return "61-90"
    elif 91 <= register_days <= 180:
        return "91-180"
    elif 181 <= register_days <= 365:
        return "181-365"
    elif 366 <= register_days <= 730:
        return "366-730"
    elif register_days > 730:
        return "730+"
    else:
        return "未定义范围"


# 根据注册天数计算预期分组
expected_register_days_range = (
    user_features["register_days"]
    .apply(get_register_days_range)
)

# 判断预期分组是否与原分组一致
register_days_range_mismatch = (
    expected_register_days_range
    != user_features["register_days_range"]
)

print(
    "注册天数与注册天数分组不一致的记录数：",
    register_days_range_mismatch.sum()
)

# 查看异常记录
register_days_range_issues = user_features.loc[
    register_days_range_mismatch,
    [
        "user_id",
        "register_days",
        "register_days_range"
    ]
].copy()

register_days_range_issues["预期注册天数分组"] = (
    expected_register_days_range.loc[
        register_days_range_mismatch
    ]
)

display(register_days_range_issues)

粉丝数与粉丝分组不一致的记录数： 0


,user_id,fans_user_num,fans_user_num_range,预期粉丝分组


关注数与关注数分组不一致的记录数： 0


,user_id,follow_user_num,follow_user_num_range,预期关注数分组


朋友数与朋友数分组不一致的记录数： 0


,user_id,friend_user_num,friend_user_num_range,预期朋友数分组


注册天数与注册天数分组不一致的记录数： 0


,user_id,register_days,register_days_range,预期注册天数分组


## 4.数据清洗


In [9]:
# 对字段["is_live_streamer"]进行清洗, 非0，1的值，全部转换成缺失值。
user_features["is_live_streamer_clean"] = (
    user_features["is_live_streamer"]
    .where(
        user_features["is_live_streamer"].isin([0, 1])
    )
    .astype("Int8")
)


In [10]:
print("原字段分布：")
print(
    user_features["is_live_streamer"]
    .value_counts(dropna=False)
)

print("\n清洗字段分布：")
print(
    user_features["is_live_streamer_clean"]
    .value_counts(dropna=False)
)

print("\n清洗后用户表的形状：")
print(user_features.shape)


原字段分布：
is_live_streamer
-124    21127
 1       6158
Name: count, dtype: int64

清洗字段分布：
is_live_streamer_clean
<NA>    21127
1        6158
Name: count, dtype: Int64

清洗后用户表的形状：
(27285, 32)


## 5. 本表检查与清洗结论

- 原表共 **27,285行、31列**；
- **user_id** 没有重复，可以作为用户特征表的主键；
- 缺失值只出现在匿名字段中：**onehot_feat4** 缺失874条，**onehot_feat12 至 onehot_feat17** 各缺失714条；
- 完整重复行为**0**，因此不删除任何用户记录
- 匿名字段的真实业务含义没有公开，因此保留原值和缺失值，不使用平均数、众数等方法擅自填补，也暂不用于核心用户分层；
- **user_active_degree** 中有6条 UNKNOWN，作为明确的未知类别保留，不删除对应用户；
- **is_lowactive_period** 全部为0，没有区分能力，保留字段，但后续不用于用户分层；
- **is_video_author** 只包含0和1，字段取值正常；
- **is_live_streamer** 原字段包含6,158条1和21,127条-124。由于-124不能解释为“不是直播用户”，新增 **is_live_streamer_clean**：合法的1予以保留，-124转换为缺失值；
- **follow_user_num、fans_user_num、friend_user_num 和 register_days** 均不存在负数；
- 粉丝数等字段存在明显长尾和极端值，但在业务上可能对应头部用户，因此不因数值较大而直接删除；
- 对 **follow_user_num 与 follow_user_num_range、fans_user_num 与 fans_user_num_range、friend_user_num 与 friend_user_num_range、register_days 与 register_days_range** 进行了跨字段逻辑一致性检查；四组字段均未发现数值与分组不一致的记录，异常记录数均为 **0**, 说明上述数值字段与对应分组字段之间的区间映射关系正确，分组边界也不存在冲突；
- 清洗后仍保留 **27,285名用户**，新增1个清洗字段，共32列，没有删除用户记录；
- 原始CSV没有被修改，后续分析继续使用内存中的 **user_features**，并优先使用 **is_live_streamer_clean** 进行相关分析。


In [11]:
processed_dir = PROCESSED_DIR
processed_dir.mkdir(parents=True, exist_ok=True)
print("清洗数据保存目录：", processed_dir.relative_to(PROJECT_ROOT).as_posix())


清洗数据保存目录： data/processed


In [12]:
user_path = (
    processed_dir
    / "user_features_clean.parquet"
)

user_features.to_parquet(
    user_path,
    index=False
)

print(
    "用户特征表已导出：",
    user_path,
    user_features.shape
)


用户特征表已导出： E:\Users\yanyan\Desktop\KuaiRand_Pure\data\processed\user_features_clean.parquet (27285, 32)
